# 8.4 中国与美国的监管 / Regulation in China and the United States

中国按服务类型分别制定规章，美国没有联邦层面的综合立法而以行政令和州法为主。两者与欧盟的做法怎样对照。

China issues separate rules by type of service, and the United States has no comprehensive federal statute and relies on executive orders and state laws. How the two compare with the approach of the European Union.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. 显式标识、隐式标识与平台核验 / Explicit labels, implicit labels and the platform check

按照《人工智能生成合成内容标识办法》的思路，实现生成一侧的两种标识和传播平台一侧的三种情形判断。元数据的字段名是本练习自拟的，实际格式由强制性国家标准规定。测试会依次模拟标识完整、元数据丢失、标识被全部删除等情形。请留意第三种情形下平台已经无从判断，这说明了为什么规定禁止删除标识。

Following the idea of the Measures for Labeling AI-Generated Synthetic Content, implement the two kinds of labels on the generating side and the distinction of three situations on the side of the distribution platform. The field names of the metadata are made up for this exercise, and the real format is laid down in a mandatory national standard. The tests simulate in turn an intact label, lost metadata and the removal of all labels. Note that in the third situation the platform has nothing left to go on, which shows why removing labels is prohibited.

In [ ]:
# China's Measures for Labeling AI-Generated Synthetic Content (in force since 1 September 2025) distinguish
#   explicit labels: visible to the user, for text a notice at the start, the end or a suitable place in between;
#   implicit labels: written into the metadata of the file, containing the attribute of the content, the name
#                    or code of the service provider, and a content number.
# Platforms that distribute content have to check the metadata and tell their users what they found.
# This exercise builds a toy version of both sides. An item is a dict {"text": ..., "metadata": {...}}.
# The field names of the metadata are made up for this exercise; the real format is laid down in a national standard.

NOTICE = "（AI生成）"


def explicit_label(text, where="end", notice=NOTICE):
    """Add the visible notice at the start (where="start") or at the end (any other value) of the text."""
    # TODO 1
    raise NotImplementedError


def implicit_label(provider, content_id):
    """The metadata of a generated item: {"generated_by_ai": True, "provider": ..., "content_id": ...}"""
    # TODO 2
    raise NotImplementedError


def make_item(text, provider, content_id, where="end"):
    """What a generation service hands out: the text with its explicit label, plus the implicit label as metadata."""
    # TODO 3
    raise NotImplementedError


def platform_check(item, user_declared=False, notice=NOTICE):
    """The three cases a distribution platform has to tell apart, checked in this order:

      "generated"           the metadata carry generated_by_ai == True
      "possibly generated"  no such metadata, but the uploading user declares the content as generated
      "suspected"           neither of the above, but the visible notice occurs in the text
      "none"                otherwise
    The item may have no "metadata" key at all, or metadata that is None.
    """
    # TODO 4
    raise NotImplementedError


def display(item, user_declared=False):
    """The text as the platform shows it, with a prefix according to platform_check:
       "[AI-generated] ", "[possibly AI-generated] ", "[suspected AI-generated] " or no prefix."""
    # TODO 5
    raise NotImplementedError

In [ ]:
# 检查 / checks
assert explicit_label("你好。") == "你好。（AI生成）" and explicit_label("你好。", "start") == "（AI生成）你好。"
assert explicit_label("Hi.", notice=" [AI]") == "Hi. [AI]"
assert implicit_label("example-llm", "c-0001") == {"generated_by_ai": True, "provider": "example-llm", "content_id": "c-0001"}

item = make_item("今天天气不错。", "example-llm", "c-0001")
assert item == {"text": "今天天气不错。（AI生成）", "metadata": {"generated_by_ai": True, "provider": "example-llm", "content_id": "c-0001"}}

# 1. The label is intact
assert platform_check(item) == "generated" and display(item).startswith("[AI-generated] ")

# 2. Someone copied the text into a new file: the metadata are gone, the visible notice is still there
copied = {"text": item["text"]}
assert platform_check(copied) == "suspected" and platform_check(copied, user_declared=True) == "possibly generated"
assert platform_check({"text": item["text"], "metadata": None}) == "suspected"

# 3. Someone removed the visible notice too. Nothing is left to detect, which is why removing labels is forbidden.
stripped = {"text": item["text"].replace(NOTICE, ""), "metadata": {}}
assert platform_check(stripped) == "none" and display(stripped) == "今天天气不错。"
assert platform_check(stripped, user_declared=True) == "possibly generated"

# 4. Only the visible notice was removed, the metadata survived
half = {"text": "今天天气不错。", "metadata": item["metadata"]}
assert platform_check(half) == "generated", "the metadata are checked first"

# 5. Ordinary content written by a person
human = {"text": "我自己写的一段话。", "metadata": {"author": "someone"}}
assert platform_check(human) == "none" and display(human) == "我自己写的一段话。"
assert display(copied) == "[suspected AI-generated] 今天天气不错。（AI生成）"
assert display(stripped, user_declared=True) == "[possibly AI-generated] 今天天气不错。"

print("m84_label: all tests passed")
for name, it, declared in (("intact", item, False), ("metadata lost", copied, False), ("all labels removed", stripped, False), ("removed, user declares", stripped, True)):
    print(f"  {name:24s} -> {platform_check(it, declared)}")

<details><summary><b>提示 1 / Hint 1</b></summary>

`platform_check` 按顺序判断，先看元数据，再看用户声明，最后看文本中是否出现显式提示。

`platform_check` decides in order, first the metadata, then the declaration of the user, and last whether the visible notice occurs in the text.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

元数据可能不存在或为 `None`，可以写成 `meta = item.get('metadata') or {}`。

The metadata may be missing or `None`, which `meta = item.get('metadata') or {}` handles.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
NOTICE = "（AI生成）"


def explicit_label(text, where="end", notice=NOTICE):
    return notice + text if where == "start" else text + notice


def implicit_label(provider, content_id):
    return {"generated_by_ai": True, "provider": provider, "content_id": content_id}


def make_item(text, provider, content_id, where="end"):
    return {"text": explicit_label(text, where), "metadata": implicit_label(provider, content_id)}


def platform_check(item, user_declared=False, notice=NOTICE):
    meta = item.get("metadata") or {}
    if meta.get("generated_by_ai") is True:
        return "generated"                       # the implicit label is present
    if user_declared:
        return "possibly generated"              # no label, but the user says so
    if notice in item.get("text", ""):
        return "suspected"                       # no label, no declaration, but a visible trace
    return "none"


def display(item, user_declared=False):
    status = platform_check(item, user_declared)
    prefix = {"generated": "[AI-generated] ", "possibly generated": "[possibly AI-generated] ",
              "suspected": "[suspected AI-generated] ", "none": ""}[status]
    return prefix + item.get("text", "")
```

</details>

## 2. 三个法域的情景对照 / Scenarios across three jurisdictions

根据 8.3 和本模块的正文，对下面每个情景分别说明在欧盟、中国和美国各自需要注意什么，并给出依据。这些判断用于练习，不构成法律意见。
(a) 一名研究者在实验室里微调一个开放权重模型，只用于论文实验，不对外提供服务。
(b) 一家公司准备上线一个面向公众的通用问答聊天产品。
(c) 一家公司开发了一个提供长期情感陪伴的虚拟角色应用。
(d) 一家公司开发了一个帮助企业筛选求职者的系统。
(e) 一家机构训练了一个新的通用模型，训练计算量约为 $3 \times 10^{25}$ 次浮点运算。

Using the text of 8.3 and of this module, state for each scenario below what has to be minded in the European Union, in China and in the United States, with the basis. These judgements are for practice and are not legal advice.
(a) A researcher fine-tunes an open-weight model in a laboratory for paper experiments alone and offers no service to others.
(b) A company is about to launch a general question-answering chat product for the public.
(c) A company develops an application with virtual characters that provide long-term emotional companionship.
(d) A company develops a system that helps employers screen job applicants.
(e) An organization trains a new general-purpose model with a training compute of about $3 \times 10^{25}$ floating-point operations.

<details><summary><b>提示 1 / Hint 1</b></summary>

(c) 在中国有一部专门的办法，自 2026 年 7 月起施行。

(c) China has dedicated measures for this, in force since July 2026.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

(e) 比较 $10^{25}$ 和 $10^{26}$ 两个门槛。

(e) Compare the two thresholds $10^{25}$ and $10^{26}$.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

(a) 欧盟方面，专为科学研究与开发目的的系统和模型不在人工智能法的适用范围之内。中国方面，未向境内公众提供服务的研发和应用不适用《生成式人工智能服务管理暂行办法》。美国方面，联邦层面没有针对这类活动的普遍规则，本模块介绍的州法针对的是特定用途的开发者和部署者。三地都仍然要遵守所用模型的许可证和个人信息方面的法律。

(b) 欧盟方面，属于有透明度风险的一级，依据第 50 条须告知用户正在与人工智能交互，并以机器可读方式标记生成的内容。中国方面，适用《生成式人工智能服务管理暂行办法》，须满足内容、训练数据和标识的要求，具有舆论属性或社会动员能力的须完成安全评估和算法备案，并按标识办法添加显式和隐式标识。美国方面，没有统一的联邦要求，需要逐州确认。

(c) 欧盟方面，至少适用第 50 条的告知义务，是否涉及其他规定取决于具体的功能和对象。中国方面，适用《人工智能拟人化互动服务管理暂行办法》，须提示用户正在与人工智能而非自然人互动，连续使用每满两小时提醒一次，不得向未成年人提供虚拟亲密关系服务，用户规模达到规定数量时须开展安全评估，同时仍适用生成式服务的一般要求。美国方面，本模块没有核实到相应的联邦规则，各州的情况需要另行查证。

(d) 欧盟方面，筛选求职者属于附件三中的就业领域，是高风险系统，相应义务自 2027 年 12 月 2 日起适用。中国方面，本模块介绍的几部规章主要针对面向公众的生成式服务，这类系统更直接相关的是《个人信息保护法》中关于自动化决策的规定，正文没有展开，需要另行查阅。美国方面，以科罗拉多州为例，就业属于 SB 26-189 所称的重大决定，自 2027 年 1 月 1 日起开发者须向部署者提供技术文档，部署者须告知求职者并在不利决定后 30 天内说明该技术所起的作用。

(e) 欧盟方面，$3 \times 10^{25}$ 超过了第 51 条 $10^{25}$ 的门槛，模型被推定具有高影响能力，除通用模型提供者的四项义务外，还须履行与系统性风险相关的义务。美国方面，按公开的跟踪资料，加利福尼亚州 SB 53 的门槛是 $10^{26}$，该模型尚未达到。中国方面，现行规章不以训练计算量作为门槛，是否需要评估和备案取决于是否向境内公众提供服务以及服务的属性。同一个模型在三个法域的定位可以不同。

---

(a) In the European Union, systems and models solely for scientific research and development are outside the scope of the AI Act. In China, development and application without services to the public in China are outside the Interim Measures for the Management of Generative AI Services. In the United States there is no general federal rule for such activity, and the state laws introduced in this module address developers and deployers of particular uses. In all three the licence of the model used and the law on personal information still have to be observed.

(b) In the European Union this is the level of transparency risk, and under Article 50 users have to be informed that they are interacting with AI and generated content marked in a machine-readable way. In China the Interim Measures for the Management of Generative AI Services apply, with their requirements on content, training data and labeling, a security assessment and algorithm filing where the service has public opinion attributes or the capacity for social mobilization, and explicit and implicit labels under the labeling measures. In the United States there is no uniform federal requirement and each state has to be checked.

(c) In the European Union at least the duty to inform of Article 50 applies, and whether other provisions are involved depends on the specific functions and the people addressed. In China the Interim Measures for the Management of Anthropomorphic AI Interaction Services apply, requiring that users be told they are interacting with AI and not a natural person, a reminder after every two hours of continuous use, no virtual intimate relationship services for minors, and a security assessment once the number of users reaches the stated levels, alongside the general requirements for generative services. For the United States this module has verified no corresponding federal rule, and the situation in the states needs separate checking.

(d) In the European Union, screening job applicants belongs to the area of employment in Annex III and is a high-risk system, with obligations applying from 2 December 2027. In China the rules introduced in this module mainly address generative services for the public, and what bears more directly on such a system are the provisions on automated decision-making in the Personal Information Protection Law, which the text does not expand on and which need to be consulted separately. In the United States, taking Colorado as an example, employment is a consequential decision in the sense of SB 26-189, and from 1 January 2027 developers have to give deployers technical documentation while deployers have to notify applicants and, within 30 days after an adverse decision, describe the role the technology played.

(e) In the European Union, $3 \times 10^{25}$ exceeds the threshold of $10^{25}$ in Article 51, the model is presumed to have high-impact capabilities, and besides the four obligations of providers of general-purpose models the obligations related to systemic risk have to be met. In the United States, according to public trackers, the threshold of SB 53 of California is $10^{26}$, which this model does not reach. In China the rules in force do not use training compute as a threshold, and whether assessment and filing are needed depends on whether a service is offered to the public in China and on its attributes. One model can be positioned differently in the three jurisdictions.

</details>